Exercici 1 · Estructura d'enllaços per etiqueta (2 punts)

Per a cada etiqueta (spam, nonspam, unlabeled) calcula:

el grau de sortida mitjà dels hosts d'aquesta etiqueta,
el percentatge del total d'arestes que surten d'hosts d'aquesta etiqueta.

Presenta-ho en una taula (pot ser un print ben format). Després respon: els hosts spam envien més o menys enllaços que els altres, i el seu pes a les arestes és proporcional al seu pes en nombre de nodes?

In [12]:
# Imports — leave as is

import gzip
import csv
import networkx as nx
import matplotlib.pyplot as plt

In [13]:
# Input file locations — you might need to change these depending on where you stored your files

INPUT_NODES_FILENAME = "data/webspam-uk2007/webspam_uk2007-nodes.csv.gz"
INPUT_EDGES_FILENAME = "data/webspam-uk2007/webspam_uk2007-edges.csv.gz"

In [14]:
hosts={}
name2id={}
with gzip.open(INPUT_NODES_FILENAME , "rt", encoding="utf-8") as input_file: #obrim el file amb la informació
    reader = csv.DictReader(input_file, delimiter=',', quotechar='"') #ens ajuda a llegir tot el csv
    for record in reader:
        nodeid=int(record["nodeid"]) #extreiem el nodeid per poder utilitzar-lo com a valor
        hosts[int(nodeid)]={"name":record['hostname'], "label":record['label']} #entrem la info del record dintre del índex nodeid
        name2id[record['hostname']]=int(nodeid) #posem en el name2id el nodeid per poder-lo utilitzar

In [16]:
# 1) Comptadors d'arestes que surten de cada etiqueta
arestes_spam = 0
arestes_nonspam = 0
arestes_unlabeled = 0
total_arestes = 0

# 2) Llegim el fitxer d'arestes una sola vegada
with gzip.open(INPUT_EDGES_FILENAME, "rt", encoding="utf-8") as input_file:
    reader = csv.DictReader(input_file, delimiter=',', quotechar='"')
    for record in reader:
        source = int(record["source"])            # node d'origen de l'aresta
        etiqueta = hosts[source]["label"]         # etiqueta del node d'origen

        if etiqueta == "spam":
            arestes_spam += 1
        elif etiqueta == "nonspam":
            arestes_nonspam += 1
        else:
            arestes_unlabeled += 1

        total_arestes += 1

# 3) Nombre total de nodes (ja tenim counts de la cel·la anterior)
total_nodes = len(hosts)

# 4) Grau de sortida mitjà = arestes de l'etiqueta / nodes de l'etiqueta
grau_spam = arestes_spam / counts["spam"]
grau_nonspam = arestes_nonspam / counts["nonspam"]
grau_unlabeled = arestes_unlabeled / counts["unlabeled"]

# 5) Percentatge d'arestes sobre el total
perc_spam = 100 * arestes_spam / total_arestes
perc_nonspam = 100 * arestes_nonspam / total_arestes
perc_unlabeled = 100 * arestes_unlabeled / total_arestes

# 6) Percentatge de nodes (per comparar)
pnodes_spam = 100 * counts["spam"] / total_nodes
pnodes_nonspam = 100 * counts["nonspam"] / total_nodes
pnodes_unlabeled = 100 * counts["unlabeled"] / total_nodes

# 7) Mostrem els resultats
print("spam:      grau mitjà =", grau_spam, "| % arestes =", perc_spam, "| % nodes =", pnodes_spam)
print("nonspam:   grau mitjà =", grau_nonspam, "| % arestes =", perc_nonspam, "| % nodes =", pnodes_nonspam)
print("unlabeled: grau mitjà =", grau_unlabeled, "| % arestes =", perc_unlabeled, "| % nodes =", pnodes_unlabeled)

spam:      grau mitjà = 59.93023255813954 | % arestes = 1.1226061713934725 | % nodes = 0.3003606073570886
nonspam:   grau mitjà = 13.68838675775092 | % arestes = 4.255350430533842 | % nodes = 4.9847636843070315
unlabeled: grau mitjà = 16.01900881300933 | % arestes = 94.62204339807269 | % nodes = 94.71487570833588


Exercici 2 · Quant PageRank acumula el spam? (3 punts)

a) Amb ALPHA = 0.91, calcula la fracció del PageRank total que acumulen els hosts spam. Després mira, entre els 100 hosts amb més PageRank, quants són spam.

b) Torna a calcular el PageRank, però ara ignorant totes les arestes que surten d'un host spam (les que hi arriben es mantenen). Calcula la mateixa fracció per als hosts spam i compara-la amb la de l'apartat a).

c) Explica en 3-4 línies el resultat: per què canvia (o no) la fracció, i què significa respecte al paper del spam en el PageRank.

Consells (sense donar la solució)
A l'apartat a) ja tens el codi del PageRank normal al quadern (PS05). Per a la fracció, suma els valors de pagerank[i] només dels nodes amb etiqueta spam. Com que el PageRank total suma 1, la fracció és directament aquesta suma.
Al b) el filtre canvia respecte al non-spam del quadern: aquí només s'exclouen les arestes on l'origen és spam, no les que hi arriben. Recorda que el filtre ha d'aparèixer a dos llocs: al càlcul del grau de sortida i al bucle del PageRank.
Tingues present què passa amb els hosts spam que, amb aquest filtre, es queden sense cap aresta de sortida.
Si t'ho vols posar fàcil, fes una funció compute_pagerank que accepti la condició del filtre.

In [ ]:
# Leave this code as-is

# Initialize id2degree dict
id2degree = {}
sum=0
N = len(hosts)
for nodeid in range(N):
    id2degree[nodeid] = 0
with gzip.open(INPUT_EDGES_FILENAME,"rt",encoding="utf-8") as fitxer: #obrim el file d'edges ara
    reader=csv.DictReader(fitxer,delimiter=',',quotechar='"')
    for record in reader:
        source=int(record["source"]) #agafem el source com a valor
        id2degree[source]+=1


In [19]:
ITERATIONS = 25
ALPHA = 0.91

pagerank = [1.0/N] * N
pagerank_aux = [0.0] * N
for iteration in range(ITERATIONS): #per cada iteració del PageRAnk
    print(f"Iteration {iteration+1} of {ITERATIONS}")
    with gzip.open(INPUT_EDGES_FILENAME, "rt", encoding="utf-8") as input_file:
        reader=csv.DictReader(input_file,delimiter=',',quotechar='"')
        for record in reader:
            source=int(record["source"]) #agafem el node d'entada igual
            destination=int(record["destination"]) #agafem també el de sortida
            pagerank_aux[destination]+=pagerank[source]/id2degree[source] # el que s'afegira al destí quan s'acabi la iteració
    pagerank_nou=[ALPHA*pagerank_aux[i]+(1-ALPHA)*(1/N) for i in range(N)] #formula bàsica del PageRank evitant nodes pou
    total=sum(pagerank_nou)
    pagerank_nou=[x/total for x in pagerank_nou] #normalitzant el nou PageRank i convertint-lo en llista pel futur
    delta=sum(abs(pagerank_nou[i]-pagerank[i]) for i in range(N)) #utilitzar la linia anterior i calcular la nova delta
    print(f"delta={delta:.4f}")
    pagerank=pagerank_nou #actualitzem per la següent iteració
    pagerank_aux=[0.0]*N #retornem tot a 0

Iteration 1 of 25
delta=1.1472
Iteration 2 of 25
delta=0.4814
Iteration 3 of 25
delta=0.2268
Iteration 4 of 25
delta=0.1259
Iteration 5 of 25
delta=0.0798
Iteration 6 of 25
delta=0.0548
Iteration 7 of 25
delta=0.0400
Iteration 8 of 25
delta=0.0302
Iteration 9 of 25
delta=0.0240
Iteration 10 of 25
delta=0.0197
Iteration 11 of 25
delta=0.0169
Iteration 12 of 25
delta=0.0153
Iteration 13 of 25
delta=0.0147
Iteration 14 of 25
delta=0.0146
Iteration 15 of 25
delta=0.0145
Iteration 16 of 25
delta=0.0145
Iteration 17 of 25
delta=0.0146
Iteration 18 of 25
delta=0.0147
Iteration 19 of 25
delta=0.0148
Iteration 20 of 25
delta=0.0150
Iteration 21 of 25
delta=0.0151
Iteration 22 of 25
delta=0.0153
Iteration 23 of 25
delta=0.0155
Iteration 24 of 25
delta=0.0157
Iteration 25 of 25
delta=0.0159


In [29]:
suma=0
total=0
for i in range(N):
    if hosts[i]["label"]=="spam":
        suma+=pagerank[i]
    total+=pagerank[i]
print(f"{100*suma/total}")
top100=sorted(range(N), key=lambda i:pagerank[i], reverse=True)[:100]
spam_al_top = [i for i in top100 if hosts[i]["label"] == "spam"]
print("Spam al top 100:", len(spam_al_top))

0.16765760068794877
Spam al top 100: 0


In [34]:
#igual que el 2 però respecte els no spam
id2nsdegree = {}
N = len(hosts)
for nodeid in range(N):
    id2nsdegree[nodeid] = 0
with gzip.open(INPUT_EDGES_FILENAME,"rt",encoding="utf-8") as fitxer:
    reader=csv.DictReader(fitxer,delimiter=',',quotechar='"')
    for record in reader:
        source=int(record["source"])
        destination=int(record["destination"])
        if hosts[source]['label'] != "spam": #Aquesta és la clau perquè no hi hagi cap spam en aquesta llista
            id2nsdegree[source]+=1
#mateix funcionament del 3 però amb els nodes nonspam només
ITERATIONS = 25
ALPHA = 0.91

nspagerank = [1.0/N] * N
nspagerank_aux = [0.0] * N
for iteration in range(ITERATIONS):
    print(f"Iteration {iteration+1} of {ITERATIONS}")
    with gzip.open(INPUT_EDGES_FILENAME, "rt", encoding="utf-8") as input_file:
        reader=csv.DictReader(input_file,delimiter=',',quotechar='"')
        for record in reader:
            source=int(record["source"])
            destination=int(record["destination"])
            if hosts[source]['label'] != "spam": #aplicant la mateixa condició abans del calcul de la auxiliar per tal de que ja no la calculi
                nspagerank_aux[destination]+=nspagerank[source]/id2nsdegree[source] # dividim per trobar el valor de la següent iteració.
    nspagerank_nou=[ALPHA*nspagerank_aux[i]+(1-ALPHA)*(1/N) for i in range(N)]
    total=sum(nspagerank_nou)
    nspagerank_nou=[x/total for x in nspagerank_nou]
    delta=sum(abs(nspagerank_nou[i]-nspagerank[i]) for i in range(N))
    print(f"delta={delta:.4f}")
    nspagerank=nspagerank_nou
    nspagerank_aux=[0.0]*N
nssuma=0
nstotal=0
for i in range(N):
    if hosts[i]["label"]=="spam":
        nssuma+=nspagerank[i]
    nstotal+=nspagerank[i]
print(f"{100*nssuma/nstotal}")


Iteration 1 of 25
delta=1.1472
Iteration 2 of 25
delta=0.4816
Iteration 3 of 25
delta=0.2267
Iteration 4 of 25
delta=0.1260
Iteration 5 of 25
delta=0.0799
Iteration 6 of 25
delta=0.0548
Iteration 7 of 25
delta=0.0401
Iteration 8 of 25
delta=0.0304
Iteration 9 of 25
delta=0.0241
Iteration 10 of 25
delta=0.0198
Iteration 11 of 25
delta=0.0170
Iteration 12 of 25
delta=0.0154
Iteration 13 of 25
delta=0.0147
Iteration 14 of 25
delta=0.0146
Iteration 15 of 25
delta=0.0146
Iteration 16 of 25
delta=0.0147
Iteration 17 of 25
delta=0.0147
Iteration 18 of 25
delta=0.0149
Iteration 19 of 25
delta=0.0150
Iteration 20 of 25
delta=0.0152
Iteration 21 of 25
delta=0.0154
Iteration 22 of 25
delta=0.0156
Iteration 23 of 25
delta=0.0158
Iteration 24 of 25
delta=0.0160
Iteration 25 of 25
delta=0.0162
0.12395159995809167


En conclusió no varia molt a causa de que el nostre graf els nodes spams no són els nmés rellevants per començar i segon que cal destacar que al no apuntar quasi ningu als spams sinó ser més els que apunten doncs varia a la baixa però no en gran mesura.